# CS406 • C3-T / FMB TEST chính thức • Baseline nhóm Mân • All-in-One

**Mục tiêu theo yêu cầu Quân:** Chốt **DCTR-C3-T** (C2 + thermal-only hard guard ngưỡng **0.30**) trước khi Test, dùng **đúng mã nguồn B3 của nhóm**, train trên **1.220 FMB official Train (gộp train/dev)**, 100 epoch × 3 seed, `last.pt`; đánh giá **280 Test và 273 Test sạch** (bỏ 7 ảnh có `leak=True` từ manifest nhóm). So trực tiếp với **SAGE và TB 3 seed** trong báo cáo baseline nhóm, không dùng điểm Dev 992/228 để kết luận Test.

**Đầu vào Kaggle:** 1) `fmb-data` có đủ Train/Test; 2) `q02b-data` có MiT-B3 pinned; 3) notebook output `CS406_2111` và SAGE Train frozen (để kiểm tra parity, tùy chọn). **Internet ON** để clone 2 repo GitHub công khai tại commit khóa. Bật **GPU T4/P100**. 1 notebook → **Save Version → Save & Run All** → **1 ZIP** `CS406_C3T_FMB_GROUP_TEST_REPORT.zip` (hoặc `_BLOCKER.zip` nếu lỗi).

**Chốt trước khi mở Test:** C3-T giữ nguyên công thức C2 trong notebook do bạn cung cấp và guard C3-T từ lần seed 0/1. **Không dùng nhãn Test trong fusion hay quyết định tham số.** Segmentation sử dụng nhãn Test chỉ ở `eval_seg.py` sau khi huấn luyện xong. Đây là kiểm tra Test một lần theo yêu cầu nhóm, không dùng để tinh chỉnh lại C3-T/C5.

**Lưu ý:** 3 seed B3 × 100 epochs cần GPU đáng kể (khoảng 7–9 giờ tùy GPU). Notebook không chạy lại SAGE/TB vì baseline nhóm đã có 3 seed khóa ngày 14/11/2026. Nếu một seed chưa xong, không công bố là so sánh đủ 3 seed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, csv, hashlib, zipfile, shutil, subprocess, traceback, time, io
import numpy as np, cv2, pandas as pd

# Cấu hình đã cố định: ĐỪNG chỉnh các ngưỡng sau khi xem Test.
SEEDS = (0, 1, 2)
C3T_THERMAL_THRESHOLD = 0.30
C3T_PARAMS = {'sigma_smooth':1.0, 'edge_percentile':90.0,
              'lambda_thermal':0.75, 'lambda_conflict':0.50, 'eps':1e-8}
FULL_TRAIN = True
RUN_SMOKE = True  # smoke 8 iterations, sau đó luôn train full nếu PASS
PARITY_SAMPLE_COUNT = 20

GROUP_SHA = '7b21b3e32aec2e864effee9b6458b59a917403f7'
SAGE_SHA = '377fdbef6b8b6ccbd1139358ad66bb41ea084dc1'
SAGE_CKPT_MD5 = '241396a51a3c5e85e360a409d9e8fe4d'
B3_INIT_SHA256 = '5670aa22de1b1848b1d715cc97d8fef77761701dedc8b3d3e44525fd71012896'

FMB_ROOT = Path('/kaggle/input/datasets/eoquiither/fmb-data/FMB')
B3_INIT = Path('/kaggle/input/datasets/eoquiither/q02b-data/sage-q02c-frozen-input/q02b/nvidia/mit-b3')
OLD_SAGE_TRAIN = Path('/kaggle/input/datasets/eoquiither/q02b-data/sage-q02c-frozen-input/q02b_runs/images/sage/train')
OLD_C2_TRAIN = Path('/kaggle/input/notebooks/lchunhhu123/cs406-2111/dctr_c2_b3_inputs/dctr_c2/train')
W = Path('/kaggle/working/CS406_C3T_FMB_GROUP_TEST')
REP = W / 'report'; REP.mkdir(parents=True, exist_ok=True)
GROUP = W / 'group_xulianh'; SAGE_REPO = W / 'SAGE_IVIF'
SEG = GROUP / 'src' / 'segmentation'
IMG = W / 'images'; RUNS = W / 'runs'
RESULT = Path('/kaggle/working/CS406_C3T_FMB_GROUP_TEST_REPORT.zip')
BLOCKER = Path('/kaggle/working/CS406_C3T_FMB_GROUP_TEST_BLOCKER.zip')

assert SEEDS == (0,1,2) and C3T_THERMAL_THRESHOLD == 0.30

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()
def md5(p):
    h=hashlib.md5()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()
def save_json(p,o):Path(p).write_text(json.dumps(o,indent=2,ensure_ascii=False,default=str),encoding='utf8')
def blocker(stage,exc):
    err={'status':'BLOCKED','stage':stage,'type':type(exc).__name__,'error':str(exc),
         'traceback':traceback.format_exc(),'time_utc':datetime.now(timezone.utc).isoformat()}
    save_json(REP/'BLOCKER.json',err)
    with zipfile.ZipFile(BLOCKER,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as z:
        for p in sorted(REP.rglob('*')):
            if p.is_file():z.write(p,arcname=str(p.relative_to(REP)))
    print('BLOCKER ZIP:',BLOCKER,flush=True)

def invoke(cmd,logname,cwd=None,env=None):
    cmd=[str(x) for x in cmd]
    logfile=REP/logname
    print('RUN:', ' '.join(cmd),flush=True)
    with logfile.open('w',encoding='utf8') as f:
        p=subprocess.Popen(cmd,cwd=str(cwd) if cwd else None,env=env,
                           stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                           text=True,encoding='utf8',errors='replace',bufsize=1)
        for line in p.stdout:
            print(line,end='',flush=True);f.write(line);f.flush()
        ret=p.wait()
    if ret:raise RuntimeError(f'Command exited {ret}; see {logfile}')
    return logfile
print('B3 official TEST with fixed C3T, 3 seeds:',SEEDS)

## 1. Preflight nguồn, dependency và manifest baseline của nhóm
Ghim đúng commit repo, đủ 1.220 Train/280 Test và 7 `leak=True`, kiểm tra checkpoint Student SAGE và MiT-B3. Nếu thiếu gì sẽ tạo `_BLOCKER.zip` **trước khi train GPU**. Phần này chỉ xem metadata/Test filenames, **không đọc Test GT**.

In [ ]:
try:
    import importlib.metadata, importlib.util
    ver=importlib.metadata.version('transformers') if importlib.util.find_spec('transformers') else None
    if ver!='4.52.4':
        print('Prepare transformers',ver,'-> 4.52.4',flush=True)
        invoke([sys.executable,'-m','pip','install','-q','transformers==4.52.4'], 'install_transformers.log')
    invoke([sys.executable,'-c',"from transformers import SegformerForSemanticSegmentation;import transformers;print('B3_IMPORT_PASS',transformers.__version__)"],
           'transformers_clean_process_check.log')
    import torch
    assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings trước Run All'
    print('GPU:',torch.cuda.get_device_name(0))
    assert B3_INIT.is_dir(), f'Missing MiT-B3 input: {B3_INIT}; attach q02b-data'
    weights=next((B3_INIT/n for n in ('model.safetensors','pytorch_model.bin') if (B3_INIT/n).is_file()),None)
    assert weights is not None and (B3_INIT/'config.json').is_file(), 'MiT-B3 weights/config missing'
    assert sha256(weights)==B3_INIT_SHA256, 'MiT-B3 pretrained SHA mismatch; do not train'
    assert FMB_ROOT.is_dir(), f'Missing FMB dataset: {FMB_ROOT}'

    def clone_locked(url,dest,pin):
        if not (dest/'.git').is_dir():
            invoke(['git','clone','-q','--depth','1',url,str(dest)],'git_'+dest.name+'.log')
        head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=str(dest),text=True).strip()
        if head!=pin:
            invoke(['git','fetch','-q','--depth','1','origin',pin], 'fetch_'+dest.name+'.log',cwd=dest)
        invoke(['git','checkout','-q','--detach',pin], 'checkout_'+dest.name+'.log',cwd=dest)
        got=subprocess.check_output(['git','rev-parse','HEAD'],cwd=str(dest),text=True).strip()
        assert got==pin,(url,got,pin)
    clone_locked('https://github.com/quan14333/xulianh.git',GROUP,GROUP_SHA)
    clone_locked('https://github.com/RollingPlain/SAGE_IVIF.git',SAGE_REPO,SAGE_SHA)
    assert (SEG/'train_seg.py').is_file() and (SEG/'eval_seg.py').is_file()
    assert (SEG/'xuat_anh.py').is_file() and (GROUP/'data/manifests/test.csv').is_file()
    src_ck=SAGE_REPO/'checkpoints/checkpoints.pt'
    assert md5(src_ck)==SAGE_CKPT_MD5,'Official SAGE checkpoint MD5 mismatch'

    # Group manifests are authoritative; read Test IDs and leak flags, NOT pixel labels.
    train_df=pd.read_csv(GROUP/'data/manifests/train.csv',dtype={'image_id':str})
    dev_df=pd.read_csv(GROUP/'data/manifests/dev.csv',dtype={'image_id':str})
    test_df=pd.read_csv(GROUP/'data/manifests/test.csv',dtype={'image_id':str})
    assert (len(train_df),len(dev_df),len(test_df))==(1045,175,280)
    train_all=pd.concat([train_df,dev_df],ignore_index=True)
    assert len(set(train_all.image_id))==1220 and len(set(test_df.image_id))==280
    assert not (set(train_all.image_id)&set(test_df.image_id)), 'Train/Test ID overlap'
    mask_leak=test_df.leak.astype(str).str.lower().eq('true')
    assert int(mask_leak.sum())==7 and int((~mask_leak).sum())==273
    assert all(str(x).split('/')[0] in ('train','test') for x in pd.concat([train_all,test_df]).vis_path)

    def locate_part(part):
        opts=list(FMB_ROOT.rglob('Visible'))
        found=[p.parent for p in opts if p.parent.name==part and (p.parent/'Infrared').is_dir() and (p.parent/'Label').is_dir()]
        assert len(found)==1,(part,found)
        return found[0]
    train_dir=locate_part('train'); test_dir=locate_part('test')
    for df,root,part in ((train_all,train_dir,'train'),(test_df,test_dir,'test')):
        for row in df.itertuples():
            iid=row.image_id
            for sub in ('Visible','Infrared','Label'):
                assert (root/sub/(iid+'.png')).is_file(),f'{part} file absent: {sub}/{iid}'
    env=os.environ.copy()
    env.update({'SAGE_FMB':str(FMB_ROOT),'SAGE_MAN':str(GROUP/'data/manifests'),
                'SAGE_ANH':str(IMG),'SAGE_RUNS':str(RUNS),
                'HF_HUB_OFFLINE':'1','TRANSFORMERS_OFFLINE':'1',
                'PYTHONUNBUFFERED':'1','OMP_NUM_THREADS':'2'})
    student=W/'sage_student';student.mkdir(parents=True,exist_ok=True)
    shutil.copy2(SAGE_REPO/'model_sub/model.py',student/'model.py')
    shutil.copy2(src_ck,student/'checkpoints.pt')
    env['SAGE_STUDENT']=str(student)

    pins={'status':'PASS','group_sha':GROUP_SHA,'sage_sha':SAGE_SHA,
          'sage_ckpt_md5':md5(src_ck),'b3_weight_sha256':sha256(weights),
          'group_train_seg_sha256':sha256(SEG/'train_seg.py'),
          'group_eval_seg_sha256':sha256(SEG/'eval_seg.py'),
          'group_xuat_anh_sha256':sha256(SEG/'xuat_anh.py'),
          'manifests':{n:sha256(GROUP/f'data/manifests/{n}.csv') for n in ('train','dev','test')},
          'count_train':1220,'count_test':280,'count_test_clean':273,
          'seeds':SEEDS,'train_epochs':100,'batch':8,'lop0':True,'gop_dev':True,
          'c3t_threshold':C3T_THERMAL_THRESHOLD}
    save_json(REP/'PREFLIGHT_PROVENANCE.json',pins)
    print('PRE-FLIGHT PASS: FMB 1220/280, group code pinned, SAGE/mit-b3 checksum matched')
except Exception as e:
    blocker('preflight',e);raise

## 2. Đóng băng công thức C2 và C3-T từ notebook gốc
Cell này ghi mã C2 **nguyên văn** từ file `cs406-2111.ipynb` đã gửi. Guard C3-T được lấy đúng từ notebook trước (thermal-only, ngưỡng 0,30). Không truy cập GT.

In [ ]:
# Exact C2 source from uploaded cs406-2111.ipynb, SHA256 verified at notebook authoring time
C2_CORE_SOURCE = '"""CS406 DCTR-C2 v0.1 — deterministic VIS edge-deficit / reliability residual.\n\nBased on Quân\'s review: C2 ONLY (no thermal residual, no conflict gate).\nInput: VIS BGR, SAGE BGR (the frozen SAGE output is generated from VIS+IR).\nNo GT access during fusion. No Dev feedback in parameter selection.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass C2Config:\n    alpha: float = 0.20  # same alpha as H1, NOT tuned on Dev\n    sigma_hpf: float = 1.20  # same Gaussian scale as H1\n    sigma_tensor: float = 1.00  # structure-tensor local coherence, predeclared\n    eps_gradient: float = 0.02  # normalized Sobel magnitude stabilization\n    eps_tensor: float = 1e-4\n    reliability_scale: float = 0.06  # smooth edge signal-to-noise surrogate\n\n\ndef sha256_file(path: Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef sobel(y: np.ndarray):\n    gx = cv2.Sobel(y.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)\n    gy = cv2.Sobel(y.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)\n    e = np.sqrt(gx * gx + gy * gy)\n    return gx, gy, e\n\n\ndef c2_fuse(vis_bgr: np.ndarray, sage_bgr: np.ndarray,\n            cfg: C2Config = C2Config(), return_masks: bool = False):\n    assert vis_bgr.dtype == np.uint8 and sage_bgr.dtype == np.uint8\n    assert vis_bgr.ndim == 3 and vis_bgr.shape == sage_bgr.shape and vis_bgr.shape[2] == 3\n    v = cv2.cvtColor(vis_bgr, cv2.COLOR_BGR2YCrCb)\n    s = cv2.cvtColor(sage_bgr, cv2.COLOR_BGR2YCrCb)\n    yv = v[:, :, 0].astype(np.float32) / 255.0\n    ys = s[:, :, 0].astype(np.float32) / 255.0\n    gx, gy, ev = sobel(yv)\n    _, _, es = sobel(ys)\n\n    # Deficit: only when VIS local gradient exceeds SAGE local gradient.\n    # No segmentation GT, class IDs or Dev-selected thresholds.\n    m_def = np.clip((ev - es) / (ev + cfg.eps_gradient), 0.0, 1.0)\n\n    # VIS reliability: local oriented-structure coherence, discounted when\n    # gradient magnitude is weak. This is a testable proxy, not learned confidence.\n    sig = cfg.sigma_tensor\n    jxx = cv2.GaussianBlur(gx * gx, (0, 0), sig)\n    jyy = cv2.GaussianBlur(gy * gy, (0, 0), sig)\n    jxy = cv2.GaussianBlur(gx * gy, (0, 0), sig)\n    coherence = np.sqrt(np.maximum(0.0, (jxx - jyy) ** 2 + 4 * jxy ** 2)) / (jxx + jyy + cfg.eps_tensor)\n    m_rel = np.clip(coherence, 0, 1) * (ev / (ev + cfg.reliability_scale))\n    mask = np.clip(m_def * m_rel, 0, 1)\n\n    blur = cv2.GaussianBlur(yv, (0, 0), cfg.sigma_hpf)\n    high = yv - blur\n    delta = cfg.alpha * mask * high\n    proposed = ys + delta\n    clipped = np.clip(proposed, 0.0, 1.0)\n    # Explicit round-to-nearest uint8 and preserve SAGE\'s own chroma.\n    y_u8 = np.rint(clipped * 255.0).astype(np.uint8)\n    fused_ycc = s.copy()\n    fused_ycc[:, :, 0] = y_u8\n    reconstructed = cv2.cvtColor(fused_ycc, cv2.COLOR_YCrCb2BGR)\n    # Preserve byte-identical original SAGE BGR wherever quantization does not change Y.\n    y_changed = (y_u8 != s[:, :, 0])\n    out = np.where(y_changed[:, :, None], reconstructed, sage_bgr)\n    assert out.dtype == np.uint8 and out.shape == sage_bgr.shape\n\n    if not return_masks:\n        return out\n    _, _, eout = sobel(y_u8.astype(np.float32)/255.0)\n    changed = np.any(out != sage_bgr, axis=2)\n    # Proxy clipping rate, not percent of pixels ruined.\n    clip_rate = float(np.mean((proposed < 0) | (proposed > 1)))\n    high_def = m_def > 0.20  # diagnostic-only threshold, never used by fusion\n    low_def = m_def < 0.05   # diagnostic-only threshold, never used by fusion\n    absy = np.abs(y_u8.astype(np.int16) - s[:,:,0].astype(np.int16)).astype(np.float32)\n    info = {\n        \'mean_deficit\':float(m_def.mean()), \'mean_reliability\':float(m_rel.mean()),\n        \'mean_effective_gate\':float(mask.mean()),\n        \'changed_rgb_pixel_fraction\':float(changed.mean()),\n        \'changed_y_pixel_fraction\':float(np.mean(absy > 0)),\n        \'mae_rgb_0_255\':float(np.mean(np.abs(out.astype(np.float32)-sage_bgr.astype(np.float32)))),\n        \'mean_abs_y_lsb\':float(absy.mean()),\n        \'change_y_lsb_deficit_region\':float(absy[high_def].mean()) if np.any(high_def) else 0.0,\n        \'change_y_lsb_nondeficit_region\':float(absy[low_def].mean()) if np.any(low_def) else 0.0,\n        \'edge_ratio_sage_to_vis\':float(es.mean()/(ev.mean()+1e-8)),\n        \'edge_ratio_c2_to_vis\':float(eout.mean()/(ev.mean()+1e-8)),\n        \'clip_rate\':clip_rate,\n        \'masked_fraction_deficit_gt_0_20\':float(high_def.mean()),\n    }\n    return out, info, (m_def, m_rel, mask, high)\n\n\ndef synthetic_self_test():\n    """CPU smoke for shape, determinism, provenance and zero-GT fusion."""\n    yy, xx = np.mgrid[:128, :192]\n    base = np.zeros((128,192,3),dtype=np.uint8)\n    base[:, :] = [25,50,90]\n    base[20:106, 30:150] = [220,190,180]\n    base[45:65, 65:110] = [10,240,90]\n    sage = cv2.GaussianBlur(base,(0,0),2.4)\n    out, info, (d,r,g,h) = c2_fuse(base,sage,return_masks=True)\n    out2=c2_fuse(base,sage)\n    assert np.array_equal(out,out2)\n    assert np.isfinite(out).all() and np.isfinite(d).all() and np.isfinite(r).all()\n    assert np.all((d>=0)&(d<=1)) and np.all((r>=0)&(r<=1)) and np.all((g>=0)&(g<=1))\n    assert info[\'changed_y_pixel_fraction\'] > 0, info\n    assert info[\'changed_rgb_pixel_fraction\'] > 0, info\n    # On an exact SAGE=VIS match, no edge deficit means no changes to the Y channel.\n    out_eq, info_eq, masks_eq=c2_fuse(base,base,return_masks=True)\n    assert np.allclose(masks_eq[0], 0, atol=1e-7)\n    assert np.array_equal(out_eq, base), \'SAGE=VIS must be byte-identical when deficit is zero\'\n    return {\'status\':\'PASS\',\'demo_changed_rgb_fraction\':info[\'changed_rgb_pixel_fraction\'],\n            \'demo_changed_y_fraction\':info[\'changed_y_pixel_fraction\']}\n\nif __name__ == \'__main__\':\n    print(json.dumps(synthetic_self_test(),indent=2))'
CORE_PATH=W/'dctr_c2_core.py'
CORE_PATH.write_text(C2_CORE_SOURCE, encoding='utf8')
assert sha256(CORE_PATH) == '7641e319a714fd44b246e3921404ece1110bc6766dd4d5bdc1c9c35c4165d621'
sys.path.insert(0,str(W))
from dctr_c2_core import C2Config,c2_fuse,synthetic_self_test
assert synthetic_self_test()['status']=='PASS'
C2_CFG = C2Config()
print('Original C2 source lock PASS:',sha256(CORE_PATH))

In [ ]:
def grad_norm(im,p):
    im=cv2.GaussianBlur(im,(0,0),float(p['sigma_smooth']))
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,ksize=3)
    gy=cv2.Sobel(im,cv2.CV_32F,0,1,ksize=3)
    mag=np.sqrt(gx*gx+gy*gy)
    scale=max(float(np.percentile(mag,p['edge_percentile'])),float(p['eps']))
    return gx,gy,mag,np.clip(mag/scale,0,1)

def signal_maps(vis,ir,p=C3T_PARAMS):
    gv=cv2.cvtColor(vis,cv2.COLOR_RGB2GRAY).astype(np.float32)/255.
    gi=cv2.cvtColor(ir,cv2.COLOR_RGB2GRAY).astype(np.float32)/255.
    vx,vy,vm,vs=grad_norm(gv,p);ix,iy,im,ins=grad_norm(gi,p)
    thermal=np.maximum(ins-vs,0)
    cos=np.clip((vx*ix+vy*iy)/(vm*im+p['eps']),-1,1)
    conflict=(1-np.abs(cos))*np.minimum(ins,vs)
    return thermal,conflict


# C3-T fixed thermal-only hard revert (same code as original C3-T seed 0/1).
def c3t_fuse(vis_bgr, ir_bgr, sage_bgr, c2_bgr):
    vis_rgb=cv2.cvtColor(vis_bgr,cv2.COLOR_BGR2RGB)
    ir_rgb=cv2.cvtColor(ir_bgr,cv2.COLOR_BGR2RGB)
    thermal,conflict=signal_maps(vis_rgb,ir_rgb)
    flag=thermal>=C3T_THERMAL_THRESHOLD
    out=c2_bgr.copy();out[flag]=sage_bgr[flag]
    assert np.array_equal(out[~flag],c2_bgr[~flag]) and np.array_equal(out[flag],sage_bgr[flag])
    return out,flag

# Synthetic guard: no GT needed, deterministic.
rng=np.random.default_rng(406)
v=rng.integers(0,256,(64,80,3),np.uint8);i=rng.integers(0,256,(64,80,3),np.uint8)
s=cv2.GaussianBlur(v,(0,0),2.3);c2=c2_fuse(v,s)
a,m=c3t_fuse(v,i,s,c2);b,m2=c3t_fuse(v,i,s,c2)
assert a.shape==v.shape and a.dtype==np.uint8 and np.array_equal(a,b) and np.array_equal(m,m2)
print('C3-T fixed candidate smoke: PASS')

## 3. Sinh ảnh SAGE chính thức và xuất đủ ảnh C3-T (Train + Test)
Dùng **`xuat_anh.py` của nhóm** và **SAGE Student checkpoint đã kiểm MD5** để tạo SAGE cho 1.500 cặp. Sau đó áp dụng C2 và C3-T đóng băng trên cùng nguồn SAGE. Chỉ đọc VIS/IR/ảnh SAGE, **không đọc file Label**. Kiểm tra tham chiếu ảnh C2 Train cũ nếu Kaggle có gắn output `CS406_2111` (chỉ QA, không tune).

In [ ]:
try:
    invoke([sys.executable,'-u',str(SEG/'xuat_anh.py'),'--bien_the','sage'],
           'export_sage_1500.log',cwd=SEG,env=env)
    sage_tr=IMG/'sage/train';sage_te=IMG/'sage/test'
    assert len(list(sage_tr.glob('*.png')))==1220 and len(list(sage_te.glob('*.png')))==280
    out_tr=IMG/'c3t/train';out_te=IMG/'c3t/test'
    out_tr.mkdir(parents=True,exist_ok=True);out_te.mkdir(parents=True,exist_ok=True)
    manifest=[];start=time.perf_counter()
    parity={'old_sage_compared':0,'old_sage_identical':0,'old_sage_mae':[],
            'old_c2_compared':0,'old_c2_identical':0,'old_c2_mae':[]}
    for part,df,dir_0,sage_dir,dst in (
            ('train',train_all,train_dir,sage_tr,out_tr),
            ('test',test_df,test_dir,sage_te,out_te)):
        ids=sorted(df.image_id.tolist())
        sample=set(ids[:PARITY_SAMPLE_COUNT]) if part=='train' else set()
        for j,iid in enumerate(ids,1):
            src=[dir_0/'Visible'/(iid+'.png'),dir_0/'Infrared'/(iid+'.png'),sage_dir/(iid+'.png')]
            vis,ir,sage=[cv2.imread(str(p),cv2.IMREAD_COLOR) for p in src]
            assert all(x is not None for x in (vis,ir,sage)),(iid,src)
            assert vis.shape==ir.shape==sage.shape==(600,800,3),(iid,vis.shape,ir.shape,sage.shape)
            c2=c2_fuse(vis,sage,C2_CFG)
            c3t,mask=c3t_fuse(vis,ir,sage,c2)
            dest=dst/(iid+'.png')
            assert cv2.imwrite(str(dest),c3t)
            assert np.array_equal(cv2.imread(str(dest)),c3t)
            if iid in sample and (OLD_SAGE_TRAIN/(iid+'.png')).is_file():
                old=cv2.imread(str(OLD_SAGE_TRAIN/(iid+'.png')))
                assert old is not None and old.shape==sage.shape
                parity['old_sage_compared']+=1
                parity['old_sage_identical']+=int(np.array_equal(sage,old))
                parity['old_sage_mae'].append(float(np.abs(sage.astype(np.int16)-old.astype(np.int16)).mean()))
            if iid in sample and (OLD_C2_TRAIN/(iid+'.png')).is_file():
                old=cv2.imread(str(OLD_C2_TRAIN/(iid+'.png')))
                assert old is not None and old.shape==c2.shape
                parity['old_c2_compared']+=1
                parity['old_c2_identical']+=int(np.array_equal(c2,old))
                parity['old_c2_mae'].append(float(np.abs(c2.astype(np.int16)-old.astype(np.int16)).mean()))
            manifest.append({'image_id':iid,'split':part,'candidate_file':str(dest.relative_to(W)),
                             'sha256_png':sha256(dest),'changed_C3T_vs_C2_pixels':int(np.any(c3t!=c2,axis=2).sum()),
                             'C2_support_pixels':int(np.any(c2!=sage,axis=2).sum()),
                             'thermal_strong_pixel_count':int(mask.sum())})
            if j%250==0 or j==len(ids):print('C3T export',part,j,'/',len(ids),flush=True)
    assert len(manifest)==1500 and len(list(out_tr.glob('*.png')))==1220 and len(list(out_te.glob('*.png')))==280
    pd.DataFrame(manifest).to_csv(REP/'C3T_IMAGE_MANIFEST_1500.csv',index=False)
    parity['old_sage_mae_mean']=float(np.mean(parity.pop('old_sage_mae'))) if parity['old_sage_compared'] else None
    parity['old_c2_mae_mean']=float(np.mean(parity.pop('old_c2_mae'))) if parity['old_c2_compared'] else None
    parity['warning']='Train source parity is diagnostic; authoritative Test source is groups official SAGE Student.'
    save_json(REP/'TRAIN_SOURCE_PARITY.json',parity)
    qa={'status':'PASS','C3T_train_images':1220,'C3T_test_images':280,
        'fixed_c2_alpha':C2_CFG.alpha,'c2_source_sha256':sha256(CORE_PATH),
        'hard_thermal_threshold':C3T_THERMAL_THRESHOLD,'no_GT_for_fusion':True,
        'export_elapsed_seconds':time.perf_counter()-start,'parity':parity}
    save_json(REP/'FUSION_QA.json',qa)
    print('Fusion export PASS (1500 C3T PNG). Train parity:',parity)
except Exception as e:
    blocker('fusion_export',e);raise

## 4. Huấn luyện C3-T với baseline nhóm (3 seed × 100 epochs)
Gọi trực tiếp `train_seg.main()` của nhóm qua tiến trình riêng, **không sửa file train_seg.py**. Chỉ ghim đường dẫn pretrained MiT-B3 từ `q02b-data` trong bộ nhớ. Cờ `--gop_dev --lop0` và thông số giống baseline nhóm. Mỗi seed có `last.pt` và `config.json`; kết quả Test chỉ được tính **sau khi mọi seed train xong**.

In [ ]:
try:
    assert all((IMG/'c3t/train'/(sid+'.png')).is_file() for sid in train_all.image_id)
    TRAIN_WRAPPER=("import os,sys;sys.path.insert(0,os.environ['CS406_GROUP_SEG']);"
                   "import train_seg;train_seg.MODEL['b3']=os.environ['CS406_B3_INIT'];train_seg.main()")
    env_train=env.copy();env_train.update({'CS406_GROUP_SEG':str(SEG),'CS406_B3_INIT':str(B3_INIT)})
    common=['--bien_the','c3t','--model','b3','--gop_dev','--lop0','--epochs','100',
            '--batch','8','--tich_luy','1','--crop','512','--lr','0.00006','--workers','2']
    if RUN_SMOKE:
        invoke([sys.executable,'-u','-c',TRAIN_WRAPPER]+common+['--seed','0','--thu','8'],
               'B3_SMOKE_8ITERS.log',cwd=SEG,env=env_train)
    if not FULL_TRAIN:
        raise RuntimeError('FULL_TRAIN=False: stopped after preflight/smoke, not a valid Test run')
    for seed in SEEDS:
        run=RUNS/f'c3t_b3_s{seed}_1220_15lop'
        if (run/'last.pt').is_file() and (run/'config.json').is_file():
            print('Existing last.pt found, skipping training',seed)
        else:
            extra=['--tiep'] if (run/'resume.pt').is_file() else []
            invoke([sys.executable,'-u','-c',TRAIN_WRAPPER]+common+['--seed',str(seed)]+extra,
                   f'B3_C3T_SEED{seed}_TRAIN.log',cwd=SEG,env=env_train)
        assert (run/'last.pt').is_file() and (run/'config.json').is_file(),(seed,run)
        cfg=json.loads((run/'config.json').read_text())
        for k,v in {'bien_the':'c3t','model':'b3','seed':seed,'gop_dev':True,'lop0':True,
                    'n_train':1220,'n_dev':0,'epochs':100,'batch':8,'crop':512,'so_lop':15}.items():
            assert cfg.get(k)==v,(seed,k,cfg.get(k),v)
        assert cfg.get('iters')==15200,(seed,cfg.get('iters'))
        save_json(REP/f'C3T_SEED{seed}_MODEL_SHA.json',{'seed':seed,'sha256_last_pt':sha256(run/'last.pt')})
    print('ALL THREE B3 TRAINING RUNS PASS (15200 iters each)')
except Exception as e:
    blocker('b3_training',e);raise

## 5. Test chính thức, so SAGE/TB 3 seed, đóng gói một ZIP
Chạy đúng `eval_seg.py` của nhóm (`last.pt`, 280 + 273 clean). So kết quả với `tong_hop.csv` khóa của Mân theo **cùng seed**; không dùng số Dev. Nếu có bảng ma trận nhầm lẫn baseline của Mân trong Input, có thể bổ sung CI sau, nhưng notebook **không tự bịa CI** chỉ từ điểm trung bình.

In [ ]:
try:
    EV_WRAPPER=("import os,sys;sys.path.insert(0,os.environ['CS406_GROUP_SEG']);"
                "import train_seg;train_seg.MODEL['b3']=os.environ['CS406_B3_INIT'];"
                "import eval_seg;eval_seg.main()")
    for seed in SEEDS:
        run=RUNS/f'c3t_b3_s{seed}_1220_15lop'
        invoke([sys.executable,'-u','-c',EV_WRAPPER,'--run',str(run),'--ckpt','last','--khong_luu_anh'],
               f'B3_C3T_SEED{seed}_TEST.log',cwd=SEG,env=env_train)
        for f in ('test_ket_qua_last.json','test_iou_lop_last.csv','test_nham_last.npz'):
            assert (run/f).is_file(),(seed,f)
        with np.load(run/'test_nham_last.npz') as d:
            assert d['conf'].shape==(280,15,15)
            assert [str(x) for x in d['image_id']]==test_df.image_id.tolist()
            assert np.array_equal(d['leak'],mask_leak.to_numpy())
    # Locked group baseline snapshot. mIoU co mat from official README/tong_hop.csv.
    locked_csv=GROUP/'reports/experiments/b3_fmb_seg/tong_hop.csv'
    g=pd.read_csv(locked_csv)
    group_ref=g[(g.ckpt=='last')&(g.so_anh==273)&(g.bien_the.isin(['sage','tb']))]
    assert len(group_ref)==6 and set(group_ref.seed)==set(SEEDS)
    rows=[]
    for seed in SEEDS:
        run=RUNS/f'c3t_b3_s{seed}_1220_15lop'
        obj=json.loads((run/'test_ket_qua_last.json').read_text())
        assert obj['gop_dev'] is True and obj['lop0'] is True and obj['ckpt']=='last'
        for n in ('280','273'):
            val=obj[n]
            assert val['so_anh']==int(n) and val['so_lop_co_mat']==13
            r={'variant':'C3T','seed':seed,'split':n,'mIoU13':float(val['mIoU_co_mat']),
               'mIoU7':float(val['mIoU_7']),'mIoU14':float(val['mIoU_14']),
               'pixel_acc':float(val['pixel_acc'])}
            for baseline in ('sage','tb'):
                b=g[(g.bien_the==baseline)&(g.ckpt=='last')&(g.seed==seed)&(g.so_anh==int(n))]
                assert len(b)==1,(seed,n,baseline)
                r[f'{baseline}_mIoU13_group']=float(b.iloc[0].mIoU_co_mat)
                r[f'delta_C3T_minus_{baseline}_pp']=r['mIoU13']-r[f'{baseline}_mIoU13_group']
            rows.append(r)
    df=pd.DataFrame(rows).sort_values(['split','seed'])
    df.to_csv(REP/'C3T_VS_GROUP_BASELINES_BY_SEED.csv',index=False)
    clean=df[df.split=='273']
    mean_c3t=clean.mIoU13.mean()
    mean_sage=clean.sage_mIoU13_group.mean()
    mean_tb=clean.tb_mIoU13_group.mean()
    summary={'status':'COMPLETE','protocol':'group official SegFormer-B3 FMB 1220-Train / 273 clean Test',
             'seeds':list(SEEDS),'mIoU13_273_mean_C3T':float(mean_c3t),
             'mIoU13_273_std_C3T':float(clean.mIoU13.std(ddof=1)),
             'mIoU13_273_mean_SAGE_group':float(mean_sage),
             'mIoU13_273_mean_TB_group':float(mean_tb),
             'delta_C3T_minus_SAGE_pp_mean':float((clean.mIoU13-clean.sage_mIoU13_group).mean()),
             'delta_C3T_minus_TB_pp_mean':float((clean.mIoU13-clean.tb_mIoU13_group).mean()),
             'beats_SAGE_in_all_seeds':bool((clean.delta_C3T_minus_sage_pp>0).all()),
             'beats_TB_in_all_seeds':bool((clean.delta_C3T_minus_tb_pp>0).all()),
             'full_test_280_reported':True,'test_clean_273_reported':True,
             'baseline_reference':'repo group pinned tong_hop.csv (rounded to 0.01pp)',
             'no_dev_tuning_after_test':True,
             'limits':['One held-out Test evaluation, should not tune method afterward',
                       'Source reference group baseline CSV 2 decimal precision; deltas approximate at 0.01pp',
                       'No 95% paired CI unless baseline per-image confusion available',
                       'FMB results not cross-dataset generalization; evaluate SemanticRT separately']}
    save_json(REP/'SUMMARY_FMB_TEST.json',summary)
    md=['# CS406 C3-T – Official FMB Test / Group baseline',
        '',f'- C3-T mean mIoU13 on 273 Test: **{mean_c3t:.4f}%** (3 seed)',
        f'- SAGE mean mIoU13 (group locked): **{mean_sage:.4f}%**',
        f'- TB mean mIoU13 (group locked): **{mean_tb:.4f}%**',
        f'- Δ C3-T−SAGE **{summary["delta_C3T_minus_SAGE_pp_mean"]:+.4f} pp**; Δ C3-T−TB **{summary["delta_C3T_minus_TB_pp_mean"]:+.4f} pp**',
        '- Exact group training/eval code pinned, 1220 train, 280 Test and 273 clean, three seeds, last.pt.',
        '- Baseline SAGE/TB values from group locked report, not rerun; CI needs group confusion matrices.',
        '- Do NOT edit C3-T threshold after seeing this Test.',
        '', '## Paired by seed (273 Test)', '',
        '```text', clean.to_string(index=False), '```', '',
        '## Limitations', '',
        '- This Test was requested explicitly by group; it is no longer a validation set for further tuning.',
        '- Cross-dataset SemanticRT evaluation remains separate.',
    ]
    (REP/'REPORT_FMB_TEST.md').write_text('\n'.join(md),encoding='utf8')
    shutil.copy2(locked_csv,REP/'GROUP_LOCKED_TONG_HOP.csv')
    shutil.copy2(CORE_PATH,REP/'DCTR_C2_SOURCE_FROZEN.py')
    # include exact C3T logic to make reproducibility auditable (alongside notebook)
    (REP/'C3T_SOURCE_DESCRIPTION.txt').write_text(
        'C3-T: c2_fuse(VIS_Bgr,SAGE_Bgr, C2Config alpha=.20 ...); '
        'thermal, conflict=signal_maps(VIS_RGB,IR_RGB, C3T_PARAMS); '
        'guard=(thermal>=0.30); output=C2; output[guard]=SAGE[guard].\n',encoding='utf8')
    with zipfile.ZipFile(RESULT,'w',zipfile.ZIP_DEFLATED,compresslevel=5) as z:
        for p in sorted(REP.rglob('*')):
            if p.is_file():z.write(p,arcname=str(p.relative_to(REP)))
        for seed in SEEDS:
            run=RUNS/f'c3t_b3_s{seed}_1220_15lop'
            for n in ('config.json','log.csv','test_ket_qua_last.json','test_iou_lop_last.csv','test_nham_last.npz'):
                z.write(run/n,arcname=f'seed{seed}/{n}')
    with zipfile.ZipFile(RESULT) as z: assert z.testzip() is None
    print('\n=== OFFICIAL GROUP TEST COMPLETE ===',json.dumps(summary,indent=2,ensure_ascii=False))
    print('ONE REPORT ZIP:',RESULT,'MB:',round(RESULT.stat().st_size/1048576,2))
    print('B3 last.pt kept separately in /kaggle/working/CS406_C3T_FMB_GROUP_TEST/runs (not inside report ZIP)')
except Exception as e:
    blocker('test_evaluation_or_report',e);raise

### Hướng dẫn sau khi chạy

- Tải đúng **`CS406_C3T_FMB_GROUP_TEST_REPORT.zip`** trong Output của Kaggle. Nếu lỗi, tải **`CS406_C3T_FMB_GROUP_TEST_BLOCKER.zip`** rồi gửi để phân tích.
- Đừng xóa `last.pt` trong `/kaggle/working/CS406_C3T_FMB_GROUP_TEST/runs` nếu cần bàn giao mô hình. ZIP báo cáo chỉ chứa mIoU/IoU, confusion matrix, log, manifest, hashes.
- Để so sánh **khoảng tin cậy 95% paired** chính xác với baseline nhóm, xin Mân thêm dataset kết quả `thanhmn123/sage-b3-fmb-ketqua` gồm `test_nham_last.npz`; không được suy ra CI từ điểm mIoU trung bình.
- Nếu kết quả Test không tăng, vẫn giữ kết quả âm để báo Quân, **không dùng Test để thử ngưỡng mới**. Muốn xác nhận tổng quát hóa thì tiến tới SemanticRT camera-held-out.